# Train HGT Paper Variants on Kaggle GPU

Notebook này train baseline `hgt_t082_k5_l3_d01.yaml` và 6 cấu hình HGT theo bài báo/công trình, sau đó nén kết quả thành một file zip tại `/kaggle/working/hgt_training_results_kaggle.zip`.

## 1. Tìm repo/dataset và copy sang `/kaggle/working`

Hãy add Kaggle Dataset chứa repo và file `data/processed/graph_artifact_3tier_t082_k5.npz`. Dataset trong `/kaggle/input` là read-only, nên notebook sẽ copy sang `/kaggle/working/nt114_hgt_work` trước khi train.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# Cach 1: Neu ban muon lay code tu GitHub, dien URL repo public vao day.
# Vi du: GITHUB_REPO_URL = 'https://github.com/username/Do-an-chuyen-nganh_NT114.git'
# Neu de trong, notebook se tu tim repo trong /kaggle/input nhu cu.
GITHUB_REPO_URL = ''
GITHUB_BRANCH = ''  # de trong neu dung default branch

WORK_DIR = Path('/kaggle/working/nt114_hgt_work')

def is_repo_root(path: Path) -> bool:
    return (
        (path / 'src' / 'graphslm_ids').exists()
        and (path / 'configs' / 'hgt_t082_k5_l3_d01.yaml').exists()
        and (path / 'pyproject.toml').exists()
    )

if WORK_DIR.exists() and is_repo_root(WORK_DIR):
    print('Work dir exists:', WORK_DIR)
elif GITHUB_REPO_URL.strip():
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    cmd = ['git', 'clone']
    if GITHUB_BRANCH.strip():
        cmd += ['--branch', GITHUB_BRANCH]
    cmd += [GITHUB_REPO_URL, str(WORK_DIR)]
    print('Cloning:', ' '.join(cmd))
    subprocess.check_call(cmd)
else:
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    repo_candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        if is_repo_root(root):
            repo_candidates.append(root)
        for config_path in root.rglob('configs/hgt_t082_k5_l3_d01.yaml'):
            candidate = config_path.parents[1]
            if is_repo_root(candidate):
                repo_candidates.append(candidate)

    repo_candidates = sorted(set(repo_candidates), key=lambda p: len(str(p)))
    print('Repo candidates:')
    for candidate in repo_candidates:
        print('-', candidate)
    assert repo_candidates, (
        'Khong tim thay repo. Hay dien GITHUB_REPO_URL hoac add Kaggle Dataset chua repo.'
    )

    SOURCE_DIR = repo_candidates[0]
    print('Copying repo:', SOURCE_DIR, '->', WORK_DIR)
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    shutil.copytree(
        SOURCE_DIR,
        WORK_DIR,
        ignore=shutil.ignore_patterns('.git', '.deps', '__pycache__', '.pytest_cache', 'outputs'),
    )

os.chdir(WORK_DIR)
print('cwd =', Path.cwd())
assert is_repo_root(Path.cwd()), 'Thu muc lam viec khong phai repo root.'

## 2. Kiểm tra GPU và dependency

In [ ]:
import sys
import subprocess
import importlib.util

required_modules = ['numpy', 'pandas', 'yaml', 'torch', 'tabulate']
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    print('Installing missing modules:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'tabulate'])
else:
    print('Core modules already available.')

import torch
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Kaggle chưa bật GPU: Notebook Settings -> Accelerator -> GPU'
print('gpu count =', torch.cuda.device_count())
print('gpu 0 =', torch.cuda.get_device_name(0))

## 3. Đảm bảo graph artifact tồn tại

Nếu repo dataset thiếu file `.npz`, notebook sẽ tìm trong toàn bộ `/kaggle/input` và copy vào `data/processed` nếu tìm thấy ở dataset khác.

In [ ]:
from pathlib import Path
import shutil

processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)
required = [
    processed_dir / 'graph_artifact_3tier_t082_k5.npz',
    processed_dir / 'graph_artifact_3tier_t082_k5.meta.json',
]

for target in required:
    if target.exists():
        continue
    matches = [p for p in Path('/kaggle/input').rglob(target.name)]
    if matches:
        print('Copying', matches[0], '->', target)
        shutil.copy2(matches[0], target)

for path in required:
    print(path, path.exists(), path.stat().st_size if path.exists() else 'missing')
    assert path.exists(), f'Missing {path}. Hãy upload file này vào Kaggle Dataset.'

configs = sorted(Path('configs/hgt_paper_variants').glob('*.yaml'))
print('variant configs =', len(configs))
assert len(configs) == 6, f'Expected 6 variant configs, found {len(configs)}'
for path in configs:
    print('-', path)

## 4. Train baseline + 6 cấu hình biến thể

Mỗi run train 150 epoch, dùng `--device cuda`. Nếu Kaggle bị ngắt và bạn chạy lại notebook, run đã có `training_summary.json` sẽ được skip.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import yaml

runs = [
    {
        'name': 'baseline_t082_k5_l3_d01',
        'config': 'configs/hgt_t082_k5_l3_d01.yaml',
        'source_name': 'Project baseline t082_k5_l3_d01',
        'citation_hint': 'Local project baseline selected by graph threshold experiment t082_k5',
    },
    {'name': 'www2020_oag_l3_h256_h8', 'config': 'configs/hgt_paper_variants/hgt_t082_k5_www2020_oag_l3_h256_h8.yaml'},
    {'name': 'pyhgt_ogbmag_l4_h512_h8', 'config': 'configs/hgt_paper_variants/hgt_t082_k5_pyhgt_ogbmag_l4_h512_h8.yaml'},
    {'name': 'graphstorm_l2_h128_h8_d05', 'config': 'configs/hgt_paper_variants/hgt_t082_k5_graphstorm_l2_h128_h8_d05.yaml'},
    {'name': 'cs224w_locomotion_l2_h128_h2', 'config': 'configs/hgt_paper_variants/hgt_t082_k5_cs224w_locomotion_l2_h128_h2.yaml'},
    {'name': 'hope_backbone_l2_h256_h8', 'config': 'configs/hgt_paper_variants/hgt_t082_k5_hope_backbone_l2_h256_h8.yaml'},
    {'name': 'gptgnn_kdd2020_l3_h400_h8', 'config': 'configs/hgt_paper_variants/hgt_t082_k5_gptgnn_kdd2020_l3_h400_h8.yaml'},
]

log_dir = Path('outputs/hgt_kaggle_logs')
log_dir.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

for run in runs:
    cfg = run['config']
    with open(cfg, 'r', encoding='utf-8') as handle:
        cfg_yaml = yaml.safe_load(handle)
    summary_path = Path(cfg_yaml['train']['output_dir']) / 'training_summary.json'
    if summary_path.exists():
        print(f"\n=== SKIP {run['name']} ===")
        print('summary exists:', summary_path)
        continue

    log_path = log_dir / f"{run['name']}.log"
    print(f"\n=== TRAIN {run['name']} ===", flush=True)
    print('config:', cfg, flush=True)
    print('log:', log_path, flush=True)

    cmd = [
        sys.executable,
        '-u',
        'src/graphslm_ids/offline_path/training/train_hgt_flow_classifier.py',
        '--config', cfg,
        '--epochs', '150',
        '--device', 'cuda',
    ]
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    if code != 0:
        raise RuntimeError(f"Run failed: {cfg}")

## 5. Tổng hợp kết quả

In [ ]:
import json
from pathlib import Path
import pandas as pd
import yaml

rows = []
for run in runs:
    with open(run['config'], 'r', encoding='utf-8') as handle:
        cfg_yaml = yaml.safe_load(handle)
    summary_path = Path(cfg_yaml['train']['output_dir']) / 'training_summary.json'
    if not summary_path.exists():
        print('Missing summary, skip:', run['name'], summary_path)
        continue

    data = json.loads(summary_path.read_text(encoding='utf-8'))
    cfg = data['config']
    exp = cfg.get('experiment', {})
    model = cfg['model']
    train = cfg['train']
    val = data['best_val_metrics']
    test = data['best_test_metrics']
    rows.append({
        'run_name': run['name'],
        'run_dir': str(summary_path.parent),
        'source_name': exp.get('source_name', run.get('source_name', '')),
        'citation_hint': exp.get('citation_hint', run.get('citation_hint', '')),
        'hidden_dim': model['hidden_dim'],
        'num_layers': model['num_layers'],
        'num_heads': model['num_heads'],
        'dropout': model['dropout'],
        'lr': train['lr'],
        'weight_decay': train['weight_decay'],
        'device': train['device'],
        'best_epoch': data['best_epoch'],
        'val_macro_f1': val['macro_f1'],
        'val_accuracy': val['accuracy'],
        'test_macro_f1': test['macro_f1'],
        'test_accuracy': test['accuracy'],
        'checkpoint': data['best_checkpoint'],
    })

df = pd.DataFrame(rows).sort_values('test_macro_f1', ascending=False)
display(df)

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'hgt_kaggle_comparison.csv'
out_md = out_dir / 'hgt_kaggle_comparison.md'
df.to_csv(out_csv, index=False)
out_md.write_text(df.to_markdown(index=False), encoding='utf-8')

print('CSV:', out_csv)
print('Markdown:', out_md)

## 6. Nén toàn bộ kết quả thành một file zip để tải về

In [ ]:
from pathlib import Path
import zipfile

bundle_path = Path('/kaggle/working/hgt_training_results_kaggle.zip')
if bundle_path.exists():
    bundle_path.unlink()

include_roots = [
    Path('outputs'),
    Path('configs/hgt_t082_k5_l3_d01.yaml'),
    Path('configs/hgt_paper_variants'),
]

def add_path(zf: zipfile.ZipFile, path: Path) -> None:
    if not path.exists():
        return
    if path.is_file():
        zf.write(path, arcname=str(path))
        return
    for item in path.rglob('*'):
        if item.is_file():
            zf.write(item, arcname=str(item))

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    for root in include_roots:
        add_path(zf, root)

print('Bundle:', bundle_path)
print('Size MB:', round(bundle_path.stat().st_size / 1024 / 1024, 2))
!ls -lh /kaggle/working/hgt_training_results_kaggle.zip